# Preparación de datos para un sistema inteligente de priorización de cobranzas

Este notebook desarrolla la fase completa de preparación de datos del proyecto académico. Incluye carga, diagnóstico estructural, limpieza, feature engineering, revisión de outliers, split por `factura_id`, pipeline de preprocesamiento, balanceamiento principal mediante `class_weight`, protocolo experimental de data augmentation y exportación de artefactos.


## 1. Librerías y configuración

In [1]:

import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, RobustScaler
from sklearn.utils.class_weight import compute_class_weight

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'project_paths.py').exists():
            return candidate
    raise FileNotFoundError('No se pudo localizar la raiz del proyecto.')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from project_paths import GENERATION_DIR, PREPARATION_DIR, ensure_artifact_dirs

ensure_artifact_dirs()
DATA_PATH = GENERATION_DIR / 'features_ml.csv'
if not DATA_PATH.exists():
    DATA_PATH = ROOT / '01_generacion' / 'data' / 'features_ml.csv'
EXPORT_DIR = PREPARATION_DIR
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TARGET_NAME = "target_mora"
RAW_TARGET_NAME = "target"

VALID_TARGET_CLASSES = {"on_time", "+30", "+60", "+90"}
VALID_CONDICION_DIAS = {30, 45, 60, 90}
RATE_COLUMNS = ["tasa_cumplimiento", "tasa_contacto_cliente", "tasa_cumpl_promesas"]

SECTOR_COLUMNS = [
    "sector_retail", "sector_manufactura", "sector_servicios", "sector_construccion",
    "sector_agro", "sector_tecnologia", "sector_salud", "sector_transporte"
]

LOG_SKEWED_COLS = [
    "monto", "monto_promedio_hist", "ratio_monto", "mora_promedio_hist",
    "mora_ultimo_tramo", "num_gestiones_factura", "dias_hasta_vence_pos",
    "dias_mora_observable", "num_no_contesta_cons", "num_promesas_rotas",
    "promesas_total", "dias_transcurridos_corte"
]

BINARY_PASSTHROUGH_COLS = [
    "tiene_garantia", "tiene_disputa_activa", "tiene_promesa_activa",
    "sin_gestion_previa", "esta_vencida_al_corte", "cliente_nuevo"
] + SECTOR_COLUMNS

COUNT_RATE_NUMERIC_COLS = [
    "condicion_dias", "antiguedad_meses", "num_facturas_prev", "tasa_cumplimiento",
    "moras_consecutivas", "tasa_contacto_cliente", "tasa_cumpl_promesas",
    "friccion_contacto", "ratio_promesas_rotas", "intensidad_gestion",
    "dias_desde_ultima_gestion"
]

CATEGORICAL_COLS = ["ultimo_resultado_enc"]


## 2. Funciones auxiliares

In [2]:

def load_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")
    df = pd.read_csv(path)
    df["fecha_corte"] = pd.to_datetime(df["fecha_corte"], errors="coerce")
    if RAW_TARGET_NAME in df.columns and EXPECTED_TARGET_NAME not in df.columns:
        df = df.rename(columns={RAW_TARGET_NAME: EXPECTED_TARGET_NAME})
    return df

def validate_dataset(df: pd.DataFrame) -> dict:
    validations = {
        "shape": list(df.shape),
        "facturas_unicas": int(df["factura_id"].nunique()),
        "clientes_unicos": int(df["cliente_id"].nunique()),
        "fecha_min": str(df["fecha_corte"].min().date()),
        "fecha_max": str(df["fecha_corte"].max().date()),
        "duplicados_exactos": int(df.duplicated().sum()),
        "duplicados_factura_corte": int(df.duplicated(subset=["factura_id", "num_corte"]).sum()),
        "fechas_nulas": int(df["fecha_corte"].isna().sum()),
        "monto_no_positivo": int((df["monto"] <= 0).sum()),
        "antiguedad_no_positiva": int((df["antiguedad_meses"] <= 0).sum()),
        "condicion_invalida": int((~df["condicion_dias"].isin(VALID_CONDICION_DIAS)).sum()),
        "target_nulos": int(df[EXPECTED_TARGET_NAME].isna().sum()),
        "clases_target_invalidas": sorted(list(set(df[EXPECTED_TARGET_NAME].dropna().unique()) - VALID_TARGET_CLASSES)),
        "nulos_por_columna": df.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_dict(),
    }
    for col in RATE_COLUMNS:
        validations[f"{col}_fuera_de_rango"] = int(((df[col] < 0) | (df[col] > 1)).sum())
    validations["filas_sector_invalido"] = int((df[SECTOR_COLUMNS].sum(axis=1) != 1).sum())
    return validations

def clean_dataset(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    df = df.copy()
    df["sin_gestion_previa"] = (
        df["dias_desde_ultima_gestion"].isna() & df["ultimo_resultado_enc"].isna()
    ).astype(int)

    df["dias_desde_ultima_gestion"] = df["dias_desde_ultima_gestion"].fillna(-1)

    df["ultimo_resultado_enc"] = (
        df["ultimo_resultado_enc"].astype("Int64").astype(str).replace("<NA>", "sin_gestion_previa")
    )
    df["ultimo_resultado_enc"] = df["ultimo_resultado_enc"].apply(
        lambda x: f"cod_{x}" if x != "sin_gestion_previa" else x
    )

    redundant = {
        "num_corte_eq_num_gestiones_factura": bool((df["num_corte"] == df["num_gestiones_factura"]).all()),
        "dias_hasta_vence_eq_condicion_menos_dias_desde_emision": bool(
            (df["dias_hasta_vence"] == (df["condicion_dias"] - df["dias_desde_emision"])).all()
        ),
    }

    cols_to_drop = []
    if redundant["num_corte_eq_num_gestiones_factura"]:
        cols_to_drop.append("num_corte")
    if redundant["dias_hasta_vence_eq_condicion_menos_dias_desde_emision"]:
        cols_to_drop.append("dias_desde_emision")

    df = df.drop(columns=cols_to_drop)

    summary = {
        "columnas_eliminadas": cols_to_drop,
        "nulos_restantes": df.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_dict(),
        **redundant,
    }
    return df, summary

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["dias_transcurridos_corte"] = df["condicion_dias"] - df["dias_hasta_vence"]
    df["esta_vencida_al_corte"] = (df["dias_hasta_vence"] < 0).astype(int)
    df["dias_mora_observable"] = np.maximum(0, -df["dias_hasta_vence"])
    df["dias_hasta_vence_pos"] = np.maximum(0, df["dias_hasta_vence"])
    df["cliente_nuevo"] = (df["num_facturas_prev"] == 0).astype(int)

    df["intensidad_gestion"] = df["num_gestiones_factura"] / np.maximum(df["dias_transcurridos_corte"] + 1, 1)
    df["friccion_contacto"] = df["num_no_contesta_cons"] / np.maximum(df["num_gestiones_factura"], 1)
    df["ratio_promesas_rotas"] = df["num_promesas_rotas"] / np.maximum(df["promesas_total"], 1)
    return df

def outlier_summary_iqr(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        s = df[col].astype(float)
        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        pct_out = ((s < lower) | (s > upper)).mean() * 100
        rows.append({
            "variable": col,
            "q1": round(float(q1), 4),
            "q3": round(float(q3), 4),
            "iqr": round(float(iqr), 4),
            "%_outliers_iqr": round(float(pct_out), 2),
            "asimetria": round(float(s.skew()), 4),
        })
    return pd.DataFrame(rows).sort_values("%_outliers_iqr", ascending=False)

def split_by_factura(df: pd.DataFrame, test_size: float = 0.20, random_state: int = 42):
    factura_target = df.groupby("factura_id")[EXPECTED_TARGET_NAME].first().reset_index()
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(
        splitter.split(factura_target[["factura_id"]], factura_target[EXPECTED_TARGET_NAME])
    )

    train_ids = factura_target.iloc[train_idx].copy()
    test_ids = factura_target.iloc[test_idx].copy()

    train_facturas = set(train_ids["factura_id"])
    test_facturas = set(test_ids["factura_id"])

    assert train_facturas.isdisjoint(test_facturas), "Hay facturas compartidas entre train y test."

    df_train = df[df["factura_id"].isin(train_facturas)].copy()
    df_test = df[df["factura_id"].isin(test_facturas)].copy()
    return df_train, df_test, train_ids, test_ids

def build_feature_list(df: pd.DataFrame) -> list[str]:
    exclude = ["factura_id", "cliente_id", "fecha_corte", EXPECTED_TARGET_NAME]
    return [c for c in df.columns if c not in exclude]

def build_preprocessor(feature_cols: list[str]) -> tuple[ColumnTransformer, dict]:
    numeric_log = [c for c in LOG_SKEWED_COLS if c in feature_cols]
    binary_passthrough = [c for c in BINARY_PASSTHROUGH_COLS if c in feature_cols]
    categorical = [c for c in CATEGORICAL_COLS if c in feature_cols]
    numeric_plain = [c for c in COUNT_RATE_NUMERIC_COLS if c in feature_cols]

    used = set(numeric_log + binary_passthrough + categorical + numeric_plain)
    remainder = [c for c in feature_cols if c not in used]

    numeric_log_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", RobustScaler())
    ])

    numeric_plain_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ])

    binary_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="sin_gestion_previa")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num_log", numeric_log_pipe, numeric_log),
            ("num_plain", numeric_plain_pipe, numeric_plain),
            ("binary", binary_pipe, binary_passthrough),
            ("cat", categorical_pipe, categorical)
        ],
        remainder="drop"
    )

    schema = {
        "numeric_log": numeric_log,
        "numeric_plain": numeric_plain,
        "binary": binary_passthrough,
        "categorical": categorical,
        "remainder_not_used": remainder
    }
    return preprocessor, schema

def compute_train_class_weights(y_train: pd.Series) -> dict:
    classes = np.array(sorted(y_train.unique()))
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    return {str(cls): float(w) for cls, w in zip(classes, weights)}

def build_augmentation_protocol(df_train: pd.DataFrame, noise_scale: float = 0.01, random_state: int = 42):
    rng = np.random.default_rng(random_state)
    minority_counts = df_train[EXPECTED_TARGET_NAME].value_counts()
    minority_class = minority_counts.idxmin()

    base = df_train[df_train[EXPECTED_TARGET_NAME] == minority_class].copy()
    if base.empty:
        return pd.DataFrame()

    numeric_aug_cols = [
        "monto", "ratio_monto", "mora_promedio_hist", "mora_ultimo_tramo",
        "monto_promedio_hist", "dias_hasta_vence", "num_promesas_rotas", "promesas_total"
    ]
    numeric_aug_cols = [c for c in numeric_aug_cols if c in base.columns]

    synthetic = base.sample(n=min(len(base), 200), replace=True, random_state=random_state).copy()

    for col in numeric_aug_cols:
        std = base[col].std(ddof=0)
        if pd.isna(std) or std == 0:
            continue
        noise = rng.normal(0, noise_scale * std, size=len(synthetic))
        synthetic[col] = synthetic[col] + noise
        if (base[col] >= 0).all():
            synthetic[col] = synthetic[col].clip(lower=0)

    synthetic["es_augmented"] = 1
    return synthetic

def export_artifacts(df_final, train_ids, test_ids, schema, validations, cleaning_summary,
                     class_weights, outlier_table, feature_cols):
    try:
        df_final.to_parquet(EXPORT_DIR / "features_ml_prepared.parquet", index=False)
    except Exception:
        df_final.to_csv(EXPORT_DIR / "features_ml_prepared.csv", index=False)

    train_ids.to_csv(EXPORT_DIR / "train_facturas_ids.csv", index=False)
    test_ids.to_csv(EXPORT_DIR / "test_facturas_ids.csv", index=False)
    outlier_table.to_csv(EXPORT_DIR / "outlier_summary.csv", index=False)

    metadata = {
        "shape_final": list(df_final.shape),
        "feature_count": len(feature_cols),
        "validations": validations,
        "cleaning_summary": cleaning_summary,
        "schema": schema,
        "class_weights": class_weights
    }
    with open(EXPORT_DIR / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)


## 3. Carga y validación estructural

In [3]:

df_raw = load_dataset(DATA_PATH)
validations = validate_dataset(df_raw)

assert validations["duplicados_exactos"] == 0
assert validations["duplicados_factura_corte"] == 0
assert validations["fechas_nulas"] == 0
assert validations["monto_no_positivo"] == 0
assert validations["antiguedad_no_positiva"] == 0
assert validations["condicion_invalida"] == 0
assert validations["filas_sector_invalido"] == 0
assert validations["target_nulos"] == 0
assert len(validations["clases_target_invalidas"]) == 0

validations


{'shape': [19671, 36],
 'facturas_unicas': 5338,
 'clientes_unicos': 200,
 'fecha_min': '2023-01-01',
 'fecha_max': '2025-07-16',
 'duplicados_exactos': 0,
 'duplicados_factura_corte': 0,
 'fechas_nulas': 0,
 'monto_no_positivo': 0,
 'antiguedad_no_positiva': 0,
 'condicion_invalida': 0,
 'target_nulos': 0,
 'clases_target_invalidas': [],
 'nulos_por_columna': {'dias_desde_ultima_gestion': 5338,
  'ultimo_resultado_enc': 5338},
 'tasa_cumplimiento_fuera_de_rango': 0,
 'tasa_contacto_cliente_fuera_de_rango': 0,
 'tasa_cumpl_promesas_fuera_de_rango': 0,
 'filas_sector_invalido': 0}

## 4. Limpieza de nulos estructurales y redundancias

In [4]:

df_clean, cleaning_summary = clean_dataset(df_raw)
cleaning_summary


{'columnas_eliminadas': ['num_corte', 'dias_desde_emision'],
 'nulos_restantes': {},
 'num_corte_eq_num_gestiones_factura': True,
 'dias_hasta_vence_eq_condicion_menos_dias_desde_emision': True}

## 5. Feature engineering y diagnóstico de outliers

En esta sección se crean variables nuevas coherentes con la lógica de cobranzas y se revisan los outliers con enfoque descriptivo. No se eliminan registros extremos de forma automática.


In [5]:

df_prepared = engineer_features(df_clean)

feature_cols = build_feature_list(df_prepared)
outlier_cols = [c for c in LOG_SKEWED_COLS if c in feature_cols]
outlier_table = outlier_summary_iqr(df_prepared, outlier_cols)

print("Shape dataset preparado:", df_prepared.shape)
outlier_table.head(10)


Shape dataset preparado: (19671, 43)


,variable,q1,q3,iqr,%_outliers_iqr,asimetria
0,monto,6641.3400,36312.9200,29671.5800,7.88,1.8134
6,dias_hasta_vence_pos,0.0000,33.0000,33.0000,7.19,1.2774
9,num_promesas_rotas,0.0000,4.0000,4.0000,4.83,1.7725
8,num_no_contesta_cons,0.0000,1.0000,1.0000,4.66,2.1549
7,dias_mora_observable,0.0000,30.0000,30.0000,4.59,1.5213
10,promesas_total,1.0000,8.0000,7.0000,2.75,1.2944
2,ratio_monto,0.5587,1.4885,0.9297,1.48,6.1936
1,monto_promedio_hist,7066.3100,42436.9300,35370.6200,0.99,1.2307
4,mora_ultimo_tramo,0.0000,35.0000,35.0000,0.41,0.9675
3,mora_promedio_hist,3.7500,42.1400,38.3900,0.19,0.5571


## 6. Split por factura y distribución del target

In [6]:

df_train, df_test, train_ids, test_ids = split_by_factura(df_prepared)

X_train = df_train[feature_cols].copy()
X_test = df_test[feature_cols].copy()
y_train = df_train[EXPECTED_TARGET_NAME].copy()
y_test = df_test[EXPECTED_TARGET_NAME].copy()

print("Shape X_train:", X_train.shape)
print("Shape X_test:", X_test.shape)
print("\nDistribución del target en train")
print(y_train.value_counts(normalize=True).round(4))
print("\nDistribución del target en test")
print(y_test.value_counts(normalize=True).round(4))


Shape X_train: (15735, 39)
Shape X_test: (3936, 39)

Distribución del target en train
target_mora
+90        0.3416
+60        0.3020
+30        0.1882
on_time    0.1682
Name: proportion, dtype: float64

Distribución del target en test
target_mora
+90        0.3305
+60        0.3082
+30        0.1908
on_time    0.1705
Name: proportion, dtype: float64


## 7. Pipeline de preprocesamiento automatizado

El pipeline separa variables numéricas sesgadas, variables numéricas de conteo o tasas, variables binarias y la variable categórica `ultimo_resultado_enc`.


In [7]:

preprocessor, schema = build_preprocessor(feature_cols)
schema


{'numeric_log': ['monto',
  'monto_promedio_hist',
  'ratio_monto',
  'mora_promedio_hist',
  'mora_ultimo_tramo',
  'num_gestiones_factura',
  'dias_hasta_vence_pos',
  'dias_mora_observable',
  'num_no_contesta_cons',
  'num_promesas_rotas',
  'promesas_total',
  'dias_transcurridos_corte'],
 'numeric_plain': ['condicion_dias',
  'antiguedad_meses',
  'num_facturas_prev',
  'tasa_cumplimiento',
  'moras_consecutivas',
  'tasa_contacto_cliente',
  'tasa_cumpl_promesas',
  'friccion_contacto',
  'ratio_promesas_rotas',
  'intensidad_gestion',
  'dias_desde_ultima_gestion'],
 'binary': ['tiene_garantia',
  'tiene_disputa_activa',
  'tiene_promesa_activa',
  'sin_gestion_previa',
  'esta_vencida_al_corte',
  'cliente_nuevo',
  'sector_retail',
  'sector_manufactura',
  'sector_servicios',
  'sector_construccion',
  'sector_agro',
  'sector_tecnologia',
  'sector_salud',
  'sector_transporte'],
 'categorical': ['ultimo_resultado_enc'],
 'remainder_not_used': ['dias_hasta_vence']}

In [8]:

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print("Shape X_train procesado:", X_train_proc.shape)
print("Shape X_test procesado:", X_test_proc.shape)


Shape X_train procesado: (15735, 47)
Shape X_test procesado: (3936, 47)


## 8. Estrategia principal de balanceamiento

La estrategia principal seleccionada para este proyecto es el uso de `class_weight`, ya que respeta mejor la estructura temporal del problema y evita introducir observaciones sintéticas en el pipeline base.


In [9]:

class_weights = compute_train_class_weights(y_train)
class_weights


{'+30': 1.3280722484807563,
 '+60': 0.8278093434343434,
 '+90': 0.7318604651162791,
 'on_time': 1.4866780045351473}

## 9. Data augmentation tabular como protocolo experimental

In [10]:

synthetic_train = build_augmentation_protocol(df_train)
synthetic_train.head()


,factura_id,cliente_id,fecha_corte,monto,condicion_dias,antiguedad_meses,tiene_garantia,sector_retail,sector_manufactura,sector_servicios,...,sin_gestion_previa,dias_transcurridos_corte,esta_vencida_al_corte,dias_mora_observable,dias_hasta_vence_pos,cliente_nuevo,intensidad_gestion,friccion_contacto,ratio_promesas_rotas,es_augmented
6172,FAC001286,CLI0062,2023-07-12,7165.561560,90,48,0,0,0,0,...,0,15,0,0,75,0,0.0625,1.0,0.000000,1
9594,FAC004007,CLI0094,2024-07-04,6917.777364,45,16,0,0,0,1,...,1,0,0,0,45,0,0.0000,0.0,0.285714,1
8210,FAC004427,CLI0080,2024-08-31,42265.669048,30,10,0,0,1,0,...,1,0,0,0,30,0,0.0000,0.0,0.000000,1
8049,FAC003392,CLI0078,2024-04-07,30181.974711,60,43,1,0,0,0,...,1,0,0,0,60,0,0.0000,0.0,0.000000,1
12061,FAC000454,CLI0123,2023-03-09,7217.788787,60,7,1,0,0,0,...,1,0,0,0,60,0,0.0000,0.0,0.000000,1


## 10. Exportación del dataset final y metadatos

In [11]:

export_artifacts(
    df_final=df_prepared,
    train_ids=train_ids,
    test_ids=test_ids,
    schema=schema,
    validations=validations,
    cleaning_summary=cleaning_summary,
    class_weights=class_weights,
    outlier_table=outlier_table,
    feature_cols=feature_cols
)

print("Artefactos exportados en:", EXPORT_DIR)


Artefactos exportados en: C:\Users\KevinUbe\Documents\Proyectos\GRUPO_3\GRUPO_3\artifacts\03_preparacion


## 11. Texto académico breve para el informe

La fase de preparación de datos se desarrolló a partir del archivo `features_ml.csv`, construido bajo una lógica de scoring dinámico por cortes temporales. En primer lugar, se verificó la integridad estructural del dataset, confirmando ausencia de duplicados exactos y consistencia en variables críticas como monto, antigüedad, tasas y codificación sectorial. Posteriormente, se realizó una limpieza semántica de los nulos estructurales asociados al corte 0, transformando la ausencia de gestiones previas en una señal explícita mediante la variable `sin_gestion_previa`.

En una segunda etapa, se controlaron redundancias determinísticas y se desarrolló un bloque de feature engineering orientado a reforzar la representación temporal y operativa del riesgo de cobranza. Entre las variables incorporadas se encuentran indicadores de vencimiento al corte, días de mora observable, intensidad de gestión, fricción de contacto y razón de promesas rotas. Los valores extremos no fueron eliminados automáticamente, ya que en este contexto pueden representar casos críticos de negocio, por lo que su tratamiento se integró al pipeline mediante transformaciones robustas.

Finalmente, el dataset se particionó por `factura_id` para evitar fuga de información entre entrenamiento y prueba, y se diseñó un pipeline automatizado de preprocesamiento con imputación, transformación logarítmica, escalado robusto y codificación categórica. La estrategia principal de balanceamiento se basó en pesos de clase calculados sobre el conjunto de entrenamiento, mientras que la data augmentation se documentó como protocolo experimental independiente y no como parte del flujo principal. Como resultado, se obtuvo un dataset preparado, trazable y metodológicamente apto para el entrenamiento de modelos supervisados posteriores.
